In [1]:
# @title basic conda setup (it will take around 30 sec- 1 min)

# Install Conda
!pip install -q condacolab
import condacolab
condacolab.install()


⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:11
🔁 Restarting kernel...


In [1]:
# @title Connect Google Drive & Install Heavy Machinery (it will ask for Google Drive permission so that it can save the output there, so give the drive permissioin ; it will take aroung 7-10 mins to run so in mean time go grab a coffee ; meanwhile it prompt you to restart session to update some package don't restart, just click on cancel) and by default the ouput is saved in the Drive with folder named : try_pipeline you can change it if you want

# Mount Google Drive immediately
from google.colab import drive
drive.mount('/content/drive')


# 1. Download the code
!git clone https://github.com/varun-bhai/batch_peptide_pipeline.git
%cd batch_peptide_pipeline

# 2. Create the destination folder in your Google Drive
!mkdir -p /content/drive/MyDrive/try_pipeline

# 3. The Magic Trick: Link the local output folder directly to your Drive
!rm -rf /content/batch_peptide_pipeline/output
# !ln -s /content/drive/MyDrive/try_pipeline /content/batch_peptide_pipeline/output

# 4. Build ETFlow Environment (Fixed Installation)
!conda create -n etflow_env python=3.10 -y

# Install basic packages FIRST so they don't get blocked
!conda run -n etflow_env pip install requests rdkit biopython

# Install a specific PyTorch version with pre-compiled wheels
!conda run -n etflow_env pip install torch==2.1.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Install torch-cluster from PyG's pre-built binary link (Prevents the build error)
!conda run -n etflow_env pip install torch-cluster==1.6.3 -f https://data.pyg.org/whl/torch-2.1.0+cu118.html

# Install ETFlow last
!conda run -n etflow_env pip install etflow

# 5. Build AlphaFold & OpenBabel
# !conda create -n af2_env python=3.10 -y
# !conda run -n af2_env pip install "colabfold[alphafold] @ git+https://github.com/sokrypton/ColabFold"
# !conda install -c conda-forge openbabel -y
# 5. Build AlphaFold & OpenBabel
!conda create -n af2_env python=3.10 -y
!conda run -n af2_env pip install "colabfold[alphafold] @ git+https://github.com/sokrypton/ColabFold"

# Add this line to force the GPU version of JAX in your af2_env!
!conda run -n af2_env pip install --upgrade "jax[cuda12_pip]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

!conda install -c conda-forge openbabel -y

# Install biopython into the base Colab environment for Step 4
!pip install biopython

import os
os.environ['MPLBACKEND'] = 'Agg'


# Create the cache directory that ETFlow expects
!mkdir -p ./cache

# Use wget with retries and a longer timeout to force the download of the checkpoint
!wget --continue --tries=10 --timeout=30 -O ./cache/drugs-o3.ckpt "https://zenodo.org/records/14226681/files/drugs-o3.ckpt?download=1"

print("Download attempt finished. Check if the file size is ~101MB.")
!ls -lh ./cache/drugs-o3.ckpt


# 7. Download AlphaFold2 / ColabFold model weights (approx 1.8 GB)
# This ensures the notebook runs automatically for anyone who uses it.
print("Downloading AlphaFold2 parameters... this may take a few minutes.")
!mkdir -p /content/batch_peptide_pipeline/colabfold_params
!conda run --no-capture-output -n af2_env python -m colabfold.download --dir /content/batch_peptide_pipeline/colabfold_params
print("✅ AlphaFold2 parameters downloaded successfully!")

Mounted at /content/drive
Cloning into 'batch_peptide_pipeline'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 61 (delta 28), reused 7 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 111.81 KiB | 5.32 MiB/s, done.
Resolving deltas: 100% (28/28), done.
/content/batch_peptide_pipeline
Channels:
 - conda-forge
Platform: linux-64
Solving environment: / - done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.2
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /usr/local/envs/etflow_env

  added / updated specs:
    - python=3.10


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 

--2026-03-10 21:49:06--  https://zenodo.org/records/14226681/files/drugs-o3.ckpt?download=1
Resolving zenodo.org (zenodo.org)... 137.138.52.235, 188.184.103.118, 188.185.43.153, ...
Connecting to zenodo.org (zenodo.org)|137.138.52.235|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 100676519 (96M) [application/octet-stream]
Saving to: ‘./cache/drugs-o3.ckpt’

./cache/drugs-o3.ck 100%[===================>]  96.01M  13.0MB/s    in 8.7s    

2026-03-10 21:49:15 (11.1 MB/s) - ‘./cache/drugs-o3.ckpt’ saved [100676519/100676519]

Download attempt finished. Check if the file size is ~101MB.
-rw-r--r-- 1 root root 97M Mar 10 21:49 ./cache/drugs-o3.ckpt
✅ AlphaFold2 parameters downloaded successfully!


In [2]:
!conda run -n af2_env pip install --upgrade "jax[cuda12]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

Looking in links: https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 65.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 MB 53.8 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.9/657.9 MB 24.3 MB/s  0:00:18
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 20.5 MB/s  0:00:15
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 93.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 57.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 MB 57.9 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 73.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 65.9 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.1/338.1 MB 37.0 MB/s  0:00:07
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.5/366.5 MB 41.9 MB/s  0:00:07
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 MB 28.2 MB/s 

In [ ]:
# @title ▶️ Run a Single Custom Sequence(if you wish to run it for a single sequence use this cell and here enter the sequence in its respective placeholder either in 1. mapformat(to understand it more visit this link : https://arxiv.org/abs/2505.03403) or in the legacy pdb format like this 2. MKA(SEP)A  ) ; the output folder will have many file, if you want final pdb file of the seq refer to final_minimized.pdb

# @markdown Fill in the text boxes below, then hit the play button! ; the job_name you will give will be the name of the folder_output in the try_pipeline folder in the Drive  :
sequence = "KETAAAK(NVA)ERQH(NLE)DS" # @param {type:"string"}
job_name = "1z3m_a" # @param {type:"string"}

print(f"🚀 Starting job: {job_name}")
print(f"🧬 Sequence: {sequence}\n")

# We use the $ symbol to pass your typed variables into the command
!python main.py --sequence "$sequence" --json modifications.json --job_name "$job_name"

print(f"\n✅ Job '{job_name}' complete! All 3D files have been saved to your Google Drive.")

🚀 Starting job: 1z3m_a
🧬 Sequence: KETAAAK(NVA)ERQH(NLE)DS


[INFO] Loaded 1 sequence from command line input

[INFO] Starting job 1/1: 1z3m_a

[INFO] 1z3m_a | Step 1/5: Parsing sequence
[INFO] Command: python /content/batch_peptide_pipeline/01_parse_input.py --sequence KETAAAK(NVA)ERQH(NLE)DS --json /content/batch_peptide_pipeline/modifications.json --fasta_out /content/batch_peptide_pipeline/output/1z3m_a/parsed_sequence.fasta --mods_out /content/batch_peptide_pipeline/output/1z3m_a/modifications.txt

[INFO] 1z3m_a | Step 2/5: Backbone prediction (af2_env)
[INFO] Command: conda run --no-capture-output -n af2_env python /content/batch_peptide_pipeline/02_run_backbone.py --fasta /content/batch_peptide_pipeline/output/1z3m_a/parsed_sequence.fasta --out_pdb /content/batch_peptide_pipeline/output/1z3m_a/backbone.pdb
[PEPstrMOD2] All model files already installed.
[PEPstrMOD2] Parameters available in: /root/.cache/colabfold/params

Running alphafold2 with MSA mode...
COMPLETE: 100% 150/150

In [ ]:
# @title ▶️ Upload FASTA and Run Batch Processing(if you wish to do batch prediction use this, it takes the fasta file as the input(like how we give in alpha fold 2 colab) when you will run this cell it will ask you to upload the .fasta file after uploading the file it do the further work)

# @markdown 1. Give your batch run and for it hit Play!
job_name = "hoho" # @param {type:"string"}

from google.colab import files
import os

print("👇 Please upload your .fasta file below:")
uploaded = files.upload()

# Get the name of the file the user just uploaded
if len(uploaded) > 0:
    fasta_file = list(uploaded.keys())[0]
    print(f"\n✅ Successfully uploaded: {fasta_file}")
    print(f"🚀 Starting batch job: {job_name}...\n")

    # Run the pipeline using the uploaded file
    !python main.py --fasta_in "$fasta_file" --json modifications.json --job_name "$job_name"

    print(f"\n🎉 Batch run '{job_name}' complete! All 3D files have been saved to your Google Drive.")
else:
    print("\n❌ No file was uploaded. Please run the cell again and select a .fasta file.")

👇 Please upload your .fasta file below:


Saving peptide_sequences.fasta to peptide_sequences.fasta

✅ Successfully uploaded: peptide_sequences.fasta
🚀 Starting batch job: hoho...


[INFO] Loaded 23 sequence(s) from /content/batch_peptide_pipeline/peptide_sequences.fasta

[INFO] Starting job 1/23: 2JOC_A

[INFO] 2JOC_A | Step 1/5: Parsing sequence
[INFO] Command: python /content/batch_peptide_pipeline/01_parse_input.py --sequence GAMGPLPPGWEKRTDSNGRVYFVNHNTRI(TPO)QWEDPRS --json /content/batch_peptide_pipeline/modifications.json --fasta_out /content/batch_peptide_pipeline/output/2JOC_A/parsed_sequence.fasta --mods_out /content/batch_peptide_pipeline/output/2JOC_A/modifications.txt

[INFO] 2JOC_A | Step 2/5: Backbone prediction (af2_env)
[INFO] Command: conda run --no-capture-output -n af2_env python /content/batch_peptide_pipeline/02_run_backbone.py --fasta /content/batch_peptide_pipeline/output/2JOC_A/parsed_sequence.fasta --out_pdb /content/batch_peptide_pipeline/output/2JOC_A/backbone.pdb
[PEPstrMOD2] All model files already

In [ ]:
# !python main.py --sequence "MKA{ptm:bromo3}A" --json modifications.json --job_name test_ptm

# !python main.py --sequence "MKA(SEP)A" --json modifications.json --job_name test_seq

# !python main.py --fasta_in input_sequences.fasta --json modifications.json --job_name my_batch_run
